load and chunk policy documents

In [ ]:
import os

policy_dir = "../data/policy_docs"
documents = []

for filename in os.listdir(policy_dir):  #find out what exists within the folder
    if filename.endswith(".md"):
        with open(os.path.join(policy_dir, filename), "r") as f:
            text = f.read()
            documents.append({"filename": filename, "text": text})

print(len(documents))
print(documents[0]["text"][:200])  # preview first 200 characters of the first doc

3
# ECOA / Regulation B — Adverse Action Notice Requirements

Under the Equal Credit Opportunity Act (ECOA) and Regulation B (12 CFR 1002.9),
creditors must provide applicants with specific reasons when


chunking by section 

In [3]:
import re

chunks = []

for doc in documents:
    sections = re.split(r'\n(?=#{1,2} )', doc["text"])  # split before each # or ## heading
    for section in sections:
        section = section.strip()
        if len(section) > 20:  # skip empty/tiny fragments
            chunks.append({"source": doc["filename"], "text": section})

print(len(chunks))
for c in chunks[:3]:
    print(c["source"], "-", c["text"][:80])

9
adverse_action_requirements.md - # ECOA / Regulation B — Adverse Action Notice Requirements

Under the Equal Cred
adverse_action_requirements.md - ## Key Requirements

- The statement of reasons must be specific and indicate th
adverse_action_requirements.md - ## Credit Scoring Systems Specifically

If a creditor bases the denial on a cred


In [4]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')

chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embedder.encode(chunk_texts)

print(chunk_embeddings.shape)

/Users/vincent/everything code/python/creditLens/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(9, 384)


sotre embedding within ChromaDB

In [5]:
import chromadb

client = chromadb.PersistentClient(path="../data/chroma_db")

collection = client.create_collection(name="credit_policy")

collection.add(
    documents=chunk_texts,
    embeddings=chunk_embeddings.tolist(),
    metadatas=[{"source": c["source"]} for c in chunks],
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(collection.count())

9


a test for retrival, using a simulated query text 

In [6]:
query_text = "applicant denied due to high debt to income ratio and low credit sub-grade from a credit scoring model"

query_embedding = embedder.encode([query_text])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3
)

for i, doc in enumerate(results["documents"][0]):
    print(f"--- Result {i+1} (source: {results['metadatas'][0][i]['source']}) ---")
    print(doc)
    print()

--- Result 1 (source: model_explainability_note.md) ---
# Note on Model-Based Credit Decisions

The CFPB has clarified (Circular 2022-03 and 2023-03) that adverse action
notice requirements apply equally to credit decisions made using complex
algorithms or AI-based models — there is no exemption for automated or
machine-learning-based underwriting systems. Creditors using such systems
must still be able to identify and disclose the specific, principal factors
that drove an individual denial decision, in the same way as a traditional
underwriting process.

This means any explanation generated from a model must:
1. Reflect the model's actual reasoning (not a generic template).
2. Identify the principal factors specific to that individual applicant.
3. Avoid vague statements like "did not meet internal criteria."

--- Result 2 (source: standard_denial_reasons.md) ---
## Income & Employment
- Income insufficient for amount of credit requested
- Excessive obligations in relation to income (